In [1]:
import pandas as pd
import mtg  # mtg.connect() opens data/mtg.duckdb read-only

with mtg.connect() as con:
    games = con.sql("SELECT * FROM games").df()
    cards = con.sql("SELECT * FROM cards").df()


- grab deck_ columns
- keep only dual color decks splash_colors main_colors
- include draft_id
- include won

In [8]:
games.shape

(931230, 1400)

In [2]:
games.head()

,expansion,event_type,draft_id,draft_time,game_time,build_index,match_number,game_number,rank,opp_rank,...,"opening_hand_Ygra, Eater of All","drawn_Ygra, Eater of All","tutored_Ygra, Eater of All","deck_Ygra, Eater of All","sideboard_Ygra, Eater of All","opening_hand_Zoraline, Cosmos Caller","drawn_Zoraline, Cosmos Caller","tutored_Zoraline, Cosmos Caller","deck_Zoraline, Cosmos Caller","sideboard_Zoraline, Cosmos Caller"
0,BLB,PremierDraft,deaa4cdcd3e84d8e8b5a0ea34a0f9d79,2024-07-30 22:30:39,2024-07-30 22:58:45,0,1,1,gold,None,...,0,0,0,1,0,0,0,0,0,0
1,BLB,PremierDraft,deaa4cdcd3e84d8e8b5a0ea34a0f9d79,2024-07-30 22:30:39,2024-07-30 23:13:12,0,2,1,gold,None,...,0,1,0,1,0,0,0,0,0,0
2,BLB,PremierDraft,deaa4cdcd3e84d8e8b5a0ea34a0f9d79,2024-07-30 22:30:39,2024-07-30 23:29:06,0,3,1,gold,None,...,0,1,0,1,0,0,0,0,0,0
3,BLB,PremierDraft,deaa4cdcd3e84d8e8b5a0ea34a0f9d79,2024-07-30 22:30:39,2024-07-30 23:46:30,0,4,1,gold,None,...,1,0,0,1,0,0,0,0,0,0
4,BLB,PremierDraft,deaa4cdcd3e84d8e8b5a0ea34a0f9d79,2024-07-30 22:30:39,2024-07-30 23:55:34,0,5,1,gold,None,...,0,0,0,1,0,0,0,0,0,0


In [3]:
games.columns

Index(['expansion', 'event_type', 'draft_id', 'draft_time', 'game_time',
       'build_index', 'match_number', 'game_number', 'rank', 'opp_rank',
       ...
       'opening_hand_Ygra, Eater of All', 'drawn_Ygra, Eater of All',
       'tutored_Ygra, Eater of All', 'deck_Ygra, Eater of All',
       'sideboard_Ygra, Eater of All', 'opening_hand_Zoraline, Cosmos Caller',
       'drawn_Zoraline, Cosmos Caller', 'tutored_Zoraline, Cosmos Caller',
       'deck_Zoraline, Cosmos Caller', 'sideboard_Zoraline, Cosmos Caller'],
      dtype='str', length=1400)

In [17]:
games.drop(columns=['expansion', 'event_type', 'draft_id', 'draft_time', 'game_time',
       'build_index', 'match_number', 'game_number', 'rank', 'opp_rank'])

,main_colors,splash_colors,on_play,num_mulligans,opp_num_mulligans,opp_colors,num_turns,won,user_n_games_bucket,user_game_win_rate_bucket,...,"opening_hand_Ygra, Eater of All","drawn_Ygra, Eater of All","tutored_Ygra, Eater of All","deck_Ygra, Eater of All","sideboard_Ygra, Eater of All","opening_hand_Zoraline, Cosmos Caller","drawn_Zoraline, Cosmos Caller","tutored_Zoraline, Cosmos Caller","deck_Zoraline, Cosmos Caller","sideboard_Zoraline, Cosmos Caller"
0,BG,WR,True,0,1,RG,5,True,50,0.54,...,0,0,0,1,0,0,0,0,0,0
1,BG,WR,True,0,0,WG,10,True,50,0.54,...,0,1,0,1,0,0,0,0,0,0
2,BG,WR,True,1,0,WB,11,False,50,0.54,...,0,1,0,1,0,0,0,0,0,0
3,BG,WR,False,0,0,BG,10,True,50,0.54,...,1,0,0,1,0,0,0,0,0,0
4,BG,WR,False,0,0,BR,6,False,50,0.54,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931225,BG,UR,True,0,0,UG,14,True,5,0.86,...,0,0,0,0,0,0,0,0,0,0
931226,BG,UR,False,0,1,WUG,6,False,5,0.86,...,0,0,0,0,0,0,0,0,0,0
931227,BG,UR,True,0,0,BR,9,True,5,0.86,...,0,0,0,0,0,0,0,0,0,0
931228,BG,UR,True,0,0,UG,6,True,5,0.86,...,0,0,0,0,0,0,0,0,0,0


In [56]:
games_copy = games.copy()

In [57]:
games_copy = games_copy.loc[games_copy["splash_colors"].isna()]
games_copy.shape

(591785, 1400)

In [58]:
games_copy = games_copy[games_copy["main_colors"].str.len() == 2]
games_copy.shape

(552432, 1400)

In [59]:
games_copy['main_colors'].value_counts()

main_colors
BG    81686
WG    79721
WB    75454
BR    66064
UG    65622
WR    62314
UB    38881
RG    32236
UR    30196
WU    20258
Name: count, dtype: int64

In [60]:
deck_cols = [c for c in games.columns if c.startswith("deck_")]
games_copy = games_copy[["draft_id", "build_index","main_colors","won"] + deck_cols]
games_copy.shape


(552432, 280)

In [61]:
games_copy.head()

,draft_id,build_index,main_colors,won,deck_Agate Assault,deck_Agate-Blade Assassin,deck_Alania's Pathmaker,"deck_Alania, Divergent Storm",deck_Artist's Talent,deck_Azure Beastbinder,...,deck_Wax-Wane Witness,deck_Wear Down,deck_Whiskerquill Scribe,deck_Whiskervale Forerunner,deck_Wick's Patrol,"deck_Wick, the Whorled Mind",deck_Wildfire Howl,deck_Wishing Well,"deck_Ygra, Eater of All","deck_Zoraline, Cosmos Caller"
15,488a8410974649eb82add4b1a6f54a75,0,BR,False,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
16,488a8410974649eb82add4b1a6f54a75,0,BR,True,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
17,488a8410974649eb82add4b1a6f54a75,0,BR,True,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
18,488a8410974649eb82add4b1a6f54a75,0,BR,False,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19,488a8410974649eb82add4b1a6f54a75,0,BR,True,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [62]:
codes, _ = pd.factorize(pd.MultiIndex.from_frame(games_copy[["draft_id", "build_index"]]))
games_copy.insert(0, "deck_id", "deck_" + pd.Series(codes + 1, index=games_copy.index).astype(str))

games_copy.shape

(552432, 281)

In [53]:
games_copy

,deck_id,draft_id,build_index,main_colors,won,deck_Agate Assault,deck_Agate-Blade Assassin,deck_Alania's Pathmaker,"deck_Alania, Divergent Storm",deck_Artist's Talent,...,deck_Wax-Wane Witness,deck_Wear Down,deck_Whiskerquill Scribe,deck_Whiskervale Forerunner,deck_Wick's Patrol,"deck_Wick, the Whorled Mind",deck_Wildfire Howl,deck_Wishing Well,"deck_Ygra, Eater of All","deck_Zoraline, Cosmos Caller"
15,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,False,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
16,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,True,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
17,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,True,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
18,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,False,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,True,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931216,deck_112721,ee40bdcd6e0248f282a7ea955958e3de,0,WU,False,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
931217,deck_112721,ee40bdcd6e0248f282a7ea955958e3de,0,WU,True,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
931218,deck_112721,ee40bdcd6e0248f282a7ea955958e3de,0,WU,True,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
931219,deck_112721,ee40bdcd6e0248f282a7ea955958e3de,0,WU,False,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [63]:
# wins and losses per deck
records = games_copy.groupby("deck_id", sort=False).agg(
    wins=("won", "sum"),
    games=("won", "size"),
)
records["losses"] = records["games"] - records["wins"]

# one row per deck: keep the first game's deck-level columns, then attach the record
decks = (
    games_copy.drop_duplicates("deck_id")[["deck_id","draft_id", "build_index","main_colors"] + deck_cols]
    .merge(records.reset_index(), on="deck_id")
)

In [64]:
decks

,deck_id,draft_id,build_index,main_colors,deck_Agate Assault,deck_Agate-Blade Assassin,deck_Alania's Pathmaker,"deck_Alania, Divergent Storm",deck_Artist's Talent,deck_Azure Beastbinder,...,deck_Whiskervale Forerunner,deck_Wick's Patrol,"deck_Wick, the Whorled Mind",deck_Wildfire Howl,deck_Wishing Well,"deck_Ygra, Eater of All","deck_Zoraline, Cosmos Caller",wins,games,losses
0,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,1,0,0,0,0,0,...,0,0,0,0,0,0,0,3,6,3
1,deck_2,41dafe993991494e9bbf2894af398d25,0,WR,0,0,0,0,0,0,...,0,0,0,0,0,0,0,6,9,3
2,deck_3,f570c6a0e89a44ff83f9301b1fa8ef1b,0,WB,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2,5,3
3,deck_4,7c5e0736075241d7a0170c535f9a0cdf,0,WR,0,0,0,0,0,0,...,1,0,0,0,0,0,0,3,6,3
4,deck_5,7d6e7ac9a9eb40808c54bd251b72837c,1,RG,0,0,0,0,0,0,...,0,0,0,0,0,0,0,5,8,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112717,deck_112718,2512cda2d2244f5c8dbbb113dfecdcda,0,WB,0,1,0,0,0,0,...,0,0,0,0,0,0,1,3,5,2
112718,deck_112719,6574128ea3ec45cbb7ea966e71c74e0b,0,WG,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,3
112719,deck_112720,4783936c8e7f4448abf42a3c46958bae,0,BR,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,3,3
112720,deck_112721,ee40bdcd6e0248f282a7ea955958e3de,0,WU,0,0,0,0,0,0,...,0,0,0,0,0,0,0,3,6,3


In [ ]:
card_cols = [c for c in decks.columns if c.startswith("deck_") and c != "deck_id"]
decks["num_cards"] = decks[card_cols].sum(axis=1)


In [94]:
decks40 = decks.loc[decks['num_cards'] == 40]

In [95]:
# one row per deck per card, with the number of copies
long = (
    decks40.melt(id_vars="deck_id", value_vars=card_cols, var_name="card", value_name="count")
    .query("count > 0")
)
long["card"] = long["card"].str.removeprefix("deck_")

# one row per copy, numbered 1-40 within each deck
copies = long.loc[long.index.repeat(long["count"]), ["deck_id", "card"]]
copies["slot"] = copies.groupby("deck_id").cumcount() + 1

# spread into card_1 ... card_40
wide = copies.pivot(index="deck_id", columns="slot", values="card")
wide.columns = [f"card_{i}" for i in wide.columns]

decks_named = decks40.drop(columns=card_cols).merge(wide.reset_index(), on="deck_id")

In [96]:
decks_named_copy = decks_named.copy()

In [97]:
decks_named_copy

,deck_id,draft_id,build_index,main_colors,wins,games,losses,num_cards,card_1,card_2,...,card_31,card_32,card_33,card_34,card_35,card_36,card_37,card_38,card_39,card_40
0,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,3,6,3,40,Agate Assault,Brazen Collector,...,Swamp,Swamp,Swamp,Swamp,Swamp,Swamp,Swamp,Swamp,Take Out the Trash,Take Out the Trash
1,deck_2,41dafe993991494e9bbf2894af398d25,0,WR,6,9,3,40,Banishing Light,Brave-Kin Duo,...,Plains,Rabbit Response,Rabid Gnaw,Raccoon Rallier,Thistledown Players,Thistledown Players,Valley Questcaller,Valley Rally,Warren Elder,Whiskerquill Scribe
2,deck_3,f570c6a0e89a44ff83f9301b1fa8ef1b,0,WB,2,5,3,40,Bandit's Talent,Builder's Talent,...,Swamp,Swamp,Swamp,Swamp,Swamp,Swamp,Swamp,Swamp,Wax-Wane Witness,Wax-Wane Witness
3,deck_4,7c5e0736075241d7a0170c535f9a0cdf,0,WR,3,6,3,40,Brambleguard Captain,Brightblade Stoat,...,Plains,Plains,Raccoon Rallier,Raccoon Rallier,Thistledown Players,Thistledown Players,Thistledown Players,Whiskerquill Scribe,Whiskerquill Scribe,Whiskervale Forerunner
4,deck_5,7d6e7ac9a9eb40808c54bd251b72837c,1,RG,5,8,3,40,Bakersbane Duo,Bakersbane Duo,...,Rust-Shield Rampager,Rust-Shield Rampager,Sazacap's Brew,Take Out the Trash,Three Tree Rootweaver,Three Tree Rootweaver,Three Tree Rootweaver,Treeguard Duo,Wandertale Mentor,Wandertale Mentor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106895,deck_112718,2512cda2d2244f5c8dbbb113dfecdcda,0,WB,3,5,2,40,Agate-Blade Assassin,Banishing Light,...,Swamp,Swamp,Swamp,Swamp,Swamp,Swamp,Uncharted Haven,Wax-Wane Witness,Wax-Wane Witness,"Zoraline, Cosmos Caller"
106896,deck_112719,6574128ea3ec45cbb7ea966e71c74e0b,0,WG,0,3,3,40,Bark-Knuckle Boxer,Brave-Kin Duo,...,Repel Calamity,Rust-Shield Rampager,Short Bow,Shrike Force,Sonar Strike,Three Tree Rootweaver,Treeguard Duo,Treetop Sentries,Warren Elder,Warren Elder
106897,deck_112720,4783936c8e7f4448abf42a3c46958bae,0,BR,0,3,3,40,Blacksmith's Talent,Bonebind Orator,...,Swamp,Swamp,Swamp,Take Out the Trash,Take Out the Trash,Teapot Slinger,Uncharted Haven,Whiskerquill Scribe,Wick's Patrol,Wick's Patrol
106898,deck_112721,ee40bdcd6e0248f282a7ea955958e3de,0,WU,3,6,3,40,Dazzling Denial,Dire Downdraft,...,Salvation Swan,Shore Up,Shoreline Looter,Shrike Force,Shrike Force,Sonar Strike,Spellgyre,Sugar Coat,Thought Shucker,Warren Elder


In [98]:
# 1. Scryfall rows for real cards only: drop token versions, keep only names that appear in this set
card_names = long["card"].unique()
bl_cards = cards[
    ~cards["layout"].isin(["token", "double_faced_token", "emblem", "art_series"])
    & cards["name"].isin(card_names)
]

# 2. attach mana value and type to each deck/card row
#    validate="many_to_one" stops with an error if a card ever matches 2 Scryfall rows
long = long.merge(
    bl_cards[["name", "mana_value", "type_line"]],
    left_on="card", right_on="name", how="left", validate="many_to_one",
).drop(columns="name")

# 3. total mana value per deck = sum of (copies x mana value)
long["is_land"] = long["type_line"].str.contains("Land")
long["mv_total"] = long["count"] * long["mana_value"]
long["land_count"] = long["count"] * long["is_land"]
long["is_creature"] = long["type_line"].str.contains("Creature")
long["creature_count"] = long["count"] * long["is_creature"]


per_deck = long.groupby("deck_id").agg(
    total_mana_value=("mv_total", "sum"),
    num_lands=("land_count", "sum"),
    num_creatures=("creature_count", "sum"),
)

per_deck["num_spells"] = long[~long["is_land"]].groupby("deck_id")["count"].sum()
per_deck["avg_mana_value"] = per_deck["total_mana_value"] / per_deck["num_spells"]

decks40 = decks40.merge(per_deck.reset_index(), on="deck_id")


In [101]:
decks40_copy = decks40.copy()

In [102]:
decks40_clean = decks40_copy.drop(columns=card_cols)

In [108]:
decks40_clean['total_mana_value'] = decks40_clean['total_mana_value'].astype('int64')

In [111]:
decks40_clean.to_csv(r"C:\Users\autum\aii600-final-project\data\mtg_data_clean.csv")